In [0]:
%run ./config

##1. Configuração

In [0]:
BRONZE_STUDIES = f"{CATALOG}.{SCHEMA}.bronze_clinical_trials"
bronze = spark.table(BRONZE_STUDIES)

##2. Schema do JSON do ClinicalTrials.gov

In [0]:
clinical_json_schema = StructType([
    StructField("protocolSection", StructType([
        StructField("identificationModule", StructType([
            StructField("nctId", StringType()),
            StructField("briefTitle", StringType())])),

        StructField("statusModule", StructType([
            StructField("overallStatus", StringType()),
            StructField("startDateStruct", StructType([
                StructField("date", StringType()),
                StructField("type", StringType())])),
            
            StructField("completionDateStruct", StructType([
                StructField("date", StringType()),
                StructField("type", StringType())]))])),

        StructField("designModule", StructType([
            StructField("studyType", StringType()),
            StructField("phases", ArrayType(StringType())),
            StructField("enrollmentInfo", StructType([
                StructField("count", LongType()),
                StructField("type", StringType())]))])),

        StructField("sponsorCollaboratorsModule", StructType([
            StructField("leadSponsor", StructType([
                StructField("name", StringType()),
                StructField("class", StringType())]))])),

        StructField("contactsLocationsModule", StructType([
            StructField("locations", ArrayType(StructType([
                StructField("facility", StringType()),
                StructField("city", StringType()),
                StructField("state", StringType()),
                StructField("country", StringType()),
                StructField("geoPoint", StructType([
                    StructField("lat", DoubleType()),
                    StructField("lon", DoubleType())]))])))])),

        StructField("armsInterventionsModule", StructType([
            StructField("interventions", ArrayType(StructType([
                StructField("type", StringType()),
                StructField("name", StringType()),
                StructField("description", StringType())])))])),

        StructField("conditionsModule", StructType([
            StructField("conditions", ArrayType(StringType()))]))]))])

##3. Ler e deduplicar a Bronze

In [0]:
latest_window = (Window.partitionBy("nct_id").orderBy(F.col("collected_at").desc()))
clinical_parsed = (bronze.withColumn("row_number",F.row_number().over(latest_window)).filter(F.col("row_number") == 1).drop("row_number")
                         .withColumn("json_data",F.from_json("payload", clinical_json_schema)))

In [0]:
#Função para datas incompletas

def parse_partial_date(column):
    return (F.when(F.length(column) == 4, F.to_date(F.concat(column, F.lit("-01-01"))))
             .when(F.length(column) == 7, F.to_date(F.concat(column, F.lit("-01"))))
             .when(F.length(column) == 10, F.to_date(column)))

In [0]:
p = F.col("json_data.protocolSection")
studies = (clinical_parsed
    .select(F.upper(F.trim(p["identificationModule"]["nctId"])).alias("nct_id"),
            F.regexp_replace(F.trim(p["identificationModule"]["briefTitle"]),r"\s+"," ").alias("study_title"),
            F.upper(p["designModule"]["studyType"]).alias("study_type"),
            F.upper(p["statusModule"]["overallStatus"]).alias("overall_status"),
            F.array_join(F.array_sort(p["designModule"]["phases"]),"|").alias("phase"),
                                      p["statusModule"]["startDateStruct"]["date"].alias("start_date_original"),
                                      p["statusModule"]["startDateStruct"]["type"].alias("start_date_type"),
                                      p["statusModule"]["completionDateStruct"]["date"].alias("completion_date_original"),
                                      p["statusModule"]["completionDateStruct"]["type"].alias("completion_date_type"),
                                      p["designModule"]["enrollmentInfo"]["count"].cast("long").alias("enrollment_count"),
                                      F.upper(p["designModule"]["enrollmentInfo"]["type"]).alias("enrollment_type"),
                                      F.regexp_replace(F.trim(p["sponsorCollaboratorsModule"]["leadSponsor"]["name"]),r"\s+"," ").alias("sponsor_name"),
                                      F.upper(p["sponsorCollaboratorsModule"]["leadSponsor"]["class"]).alias("sponsor_class"),"ingestion_id","collected_at")
    .withColumn("start_date",parse_partial_date(F.col("start_date_original")))
    .withColumn("completion_date",parse_partial_date(F.col("completion_date_original")))
    .withColumn("duration_days",F.when(F.col("completion_date") >= F.col("start_date"),F.datediff("completion_date", "start_date"))))

In [0]:
studies.write.format("delta")
             .mode("overwrite")
             .option("overwriteSchema", "true")
             .saveAsTable(f"{CATALOG}.{SCHEMA}.slv_studies")